In [0]:
import os
import json

is_job = False
try:
    raw = dbutils.jobs.taskValues.get("join_customers_and_orders_data", "metadata", None)
    print("Notebook is running in Job mode")
    print(f"raw ---> {raw}")

    # get data from the json output of the job ingest_customers_data
    meta = json.loads(raw)
    catalog          = meta.get("catalog")
    schema           = meta.get("schema")
    table_name       = meta.get("table_name")

    silver_table_name = dbutils.widgets.get("silver_table_name")
    state = dbutils.widgets.get("state")
    print(f"Runtime-Job-config ---> silver_table_name: {silver_table_name}, state: {state}")
    is_job = True
except:
    is_job = False
    

In [0]:
# add n new column called is_large_order where the order will be considered large if no_of_orders > 3
if is_job:
    spark.sql(f"""
                CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{silver_table_name} (
                    customer_id STRING,
                    first_name STRING,
                    last_name STRING,
                    phone STRING,
                    order_id STRING,
                    order_date DATE,
                    product STRING,
                    category STRING,
                    quantity INT,
                    order_amount DOUBLE,
                    state STRING,
                    is_large_order BOOLEAN
                )
                USING DELTA;
              """)
    df = spark.sql(
        f"""
        SELECT
            customer_id,
            first_name,
            last_name,
            phone,
            order_id,
            TO_DATE(order_date, 'yyyy-MM-dd') AS order_date,
            product,
            category,
            CAST(quantity AS INT) AS quantity,
            CAST(order_amount AS DOUBLE) AS order_amount,
            state,
            CASE WHEN CAST(quantity AS INT) > 3 THEN TRUE ELSE FALSE END AS is_large_order
        FROM {catalog}.{schema}.{table_name}
        WHERE state = '{state}'
        """
    )
    # Append to existing silver table safely
    df.write.format("delta").mode("append").saveAsTable(
        f"{catalog}.{schema}.{silver_table_name}"
    )

    display(df)
else:
    print(f"Notebook is running in interactive mode skipping transformation of bronze table to silver table!!!")